# Physics-Informed Diffusion-PINN MRI Reconstruction
## Notebook 02 — K-Space Inspection & Inverse FFT Reconstruction

This notebook demonstrates the fundamental MRI physics workflow on official NYU fastMRI single-coil knee data:

$$\text{Raw K-Space Data } \mathbf{y} \xrightarrow{\mathcal{F}^{-1}} \text{Inverse 2D Discrete Fourier Transform} \xrightarrow{} \text{Magnitude MRI Image } \mathbf{x}$$

We inspect raw HDF5 dataset slices, visualize log-magnitude k-space frequency matrices, perform orthogonal Inverse FFT, normalize image values, and save plot artifacts.

In [ ]:
# STEP 1 — Imports & Path Verification
import os
import sys
from pathlib import Path

# Determine repository root reliably across Kaggle interactive, nbconvert, and local runs
CANDIDATE_ROOTS = [
    Path("/kaggle/working/mri-diffusion-pinn"),
    Path(os.getcwd()),
    Path(os.getcwd()).parent,
]

REPO_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / "configs" / "base.yaml").exists() and (candidate / "src").exists():
        REPO_ROOT = candidate.resolve()
        break

if REPO_ROOT is None:
    REPO_ROOT = Path("/kaggle/working/mri-diffusion-pinn")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if os.path.exists(REPO_ROOT):
    os.chdir(REPO_ROOT)

print(f"Repository Root set to : {REPO_ROOT}")
print(f"Current Working Dir    : {os.getcwd()}")

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

from src.physics.fft import kspace_to_image_torch, image_to_kspace_torch
from src.data.preprocessing import complex_abs, normalize_zero_one

# Load base configuration
with open("configs/base.yaml", "r") as f:
    config = yaml.safe_load(f)

val_dir_cfg = config.get("data", {}).get("val_dir")
if val_dir_cfg and Path(val_dir_cfg).exists():
    data_dir = Path(val_dir_cfg)
else:
    root_cfg = config.get("data", {}).get("dataset_root", "/kaggle/working/data/fastmri")
    data_dir = Path(root_cfg) if Path(root_cfg).exists() else Path("data/fastmri")

print(f"Data Directory         : {data_dir.resolve()}")

In [ ]:
# STEP 2 — Discover and Inspect HDF5 Files
h5_files = sorted(list(data_dir.glob("*.h5")) + list(data_dir.glob("*.hdf5")) + list(data_dir.glob("**/*.h5")))
if not h5_files:
    raise FileNotFoundError(f"No .h5 files found in {data_dir}. Please run dataset downloader first.")

sample_file = h5_files[0]
print(f"Found {len(h5_files)} HDF5 files. Selecting sample: {sample_file.name}")

with h5py.File(sample_file, "r") as hf:
    print("\nHDF5 Internal Keys:", list(hf.keys()))
    kspace_dataset = hf["kspace"]
    print(f"K-Space Shape      : {kspace_dataset.shape} (slices, height, width)")
    print(f"K-Space Data Type  : {kspace_dataset.dtype}")
    
    has_target = "reconstruction_rss" in hf or "reconstruction_esc" in hf
    print(f"Reference Image    : {'Available' if has_target else 'Not present'}")

In [ ]:
# STEP 3 — Read a Single Slice and Convert to Complex PyTorch Tensor
slice_index = kspace_dataset.shape[0] // 2  # Pick middle slice

with h5py.File(sample_file, "r") as hf:
    raw_kspace_slice = hf["kspace"][slice_index]
    target_slice = hf["reconstruction_esc"][slice_index] if "reconstruction_esc" in hf else (
        hf["reconstruction_rss"][slice_index] if "reconstruction_rss" in hf else None
    )

kspace_tensor = torch.from_numpy(raw_kspace_slice)
print(f"Selected Slice Index: {slice_index}")
print(f"K-Space Tensor Shape: {kspace_tensor.shape}")
print(f"Complex Tensor Dtype: {kspace_tensor.dtype}")

In [ ]:
# STEP 4 — Perform Inverse Fast Fourier Transform (IFFT)
# Transform from frequency domain (k-space) to spatial domain (image)
complex_image = kspace_to_image_torch(kspace_tensor)
magnitude_image = complex_abs(complex_image).numpy()

# Normalize to [0, 1]
norm_image = normalize_zero_one(magnitude_image)

print(f"Reconstructed Image Shape: {norm_image.shape}")
print(f"Image Intensity Range    : Min={norm_image.min():.4f}, Max={norm_image.max():.4f}")

In [ ]:
# STEP 5 — Numerical Verification (Forward FFT Back to K-Space)
# Forward FFT of reconstructed image should match original k-space
recomputed_kspace = image_to_kspace_torch(complex_image)
kspace_diff = torch.norm(kspace_tensor - recomputed_kspace) / torch.norm(kspace_tensor)

print(f"Relative K-space Roundtrip Reconstruction Error: {kspace_diff.item():.2e}")
assert kspace_diff.item() < 1e-4, "IFFT <-> FFT roundtrip sanity check failed!"

In [ ]:
# STEP 6 — Visualization & Save Artifact
os.makedirs("results/figures", exist_ok=True)

# Compute log-scale magnitude for k-space display
kspace_mag = np.abs(raw_kspace_slice)
kspace_log_mag = np.log(kspace_mag + 1e-9)

fig, axes = plt.subplots(1, 3 if target_slice is not None else 2, figsize=(14, 5))

# Plot 1: K-Space Log-Magnitude
im0 = axes[0].imshow(kspace_log_mag, cmap="gray")
axes[0].set_title("Raw K-Space Magnitude (Log Scale)")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Plot 2: IFFT Magnitude MRI Reconstruction
im1 = axes[1].imshow(norm_image, cmap="gray")
axes[1].set_title("IFFT Reconstructed MRI Image")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Plot 3: Reference Target (if present)
if target_slice is not None:
    target_norm = normalize_zero_one(target_slice)
    im2 = axes[2].imshow(target_norm, cmap="gray")
    axes[2].set_title("Ground Truth Reference (HDF5)")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
fig_save_path = "results/figures/kspace_ifft_inspection.png"
plt.savefig(fig_save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Inspection figure saved to: {fig_save_path}")